In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
from lightgbm import LGBMRegressor
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

DATA_DIR = Path(os.getenv("HULL_DATA_DIR", "/kaggle/input/hull-tactical-market-prediction")).expanduser()
if "HULL_DATA_DIR" not in os.environ and not DATA_DIR.is_dir():
    DATA_DIR = DATA_DIR.parent / "competitions" / DATA_DIR.name

TARGET = "market_forward_excess_returns"
TRAIN_DAYS = 800
HOLDOUT_DAYS = 0

In [ ]:
def market_factors(frame: pd.DataFrame) -> pd.DataFrame:
    factors = frame.filter(regex=r"^(?:MOM|[DEIMPSV])[0-9]+$")
    return factors.apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)


def add_interactions(factors: pd.DataFrame) -> pd.DataFrame:
    x = factors.copy()
    if {"I1", "I2"}.issubset(x):
        x["U1"] = x["I2"] - x["I1"]
    if {"M11", "I2", "I7", "I9"}.issubset(x):
        mean_rate = (x["I2"] + x["I7"] + x["I9"]) / 3.0
        x["U2"] = x["M11"] / mean_rate.replace(0, np.nan)
    for left, right in (("V1", "S1"), ("M11", "V1"), ("I9", "S1")):
        if left in x and right in x:
            x[f"{left}_{right}"] = x[left] * x[right]
    return x.replace([np.inf, -np.inf], np.nan)

In [ ]:
class MarketStrategy:
    def __init__(self, history: pd.DataFrame):
        window = history.tail(TRAIN_DAYS)
        factors = market_factors(window)
        self.columns = factors.columns[factors.notna().mean() >= 0.5]
        if self.columns.empty:
            raise ValueError("The training window has no usable market factors.")

        x = add_interactions(factors[self.columns])
        rates = self.columns[self.columns.str.startswith("I")]
        # Fill rate gaps within the training window only.
        x[rates] = x[rates].ffill().bfill()
        self.medians = x.median().fillna(0.0)
        self.scaler = StandardScaler().set_output(transform="pandas")
        x = self.scaler.fit_transform(x.fillna(self.medians))
        y = window[TARGET]

        self.xgb = XGBRegressor(
            objective="reg:squarederror", n_estimators=300, max_depth=6,
            learning_rate=0.05, random_state=42, n_jobs=1,
        )
        self.lgb = LGBMRegressor(
            objective="regression", n_estimators=200, learning_rate=0.03,
            random_state=42, n_jobs=1, verbosity=-1,
        )
        self.xgb.fit(x, y)
        self.lgb.fit(x, y)
        self.last_day = None
        self.position = 0.0

    def forecast(self, rows: pd.DataFrame) -> np.ndarray:
        factors = market_factors(rows).reindex(columns=self.columns)
        x = add_interactions(factors).fillna(self.medians)
        x = self.scaler.transform(x)
        return (self.xgb.predict(x) + self.lgb.predict(x)) / 2.0

    def predict(self, day: pl.DataFrame) -> float:
        if day.height != 1:
            raise ValueError("Expected one trading day per call.")
        date_id = day["date_id"][0]
        if date_id == self.last_day:
            return self.position
        if self.last_day is not None and date_id < self.last_day:
            raise ValueError("Trading days must arrive in chronological order.")

        estimate = self.forecast(day.to_pandas())[0]
        target_position = np.clip(500.0 * estimate, 0.0, 2.0)
        self.position = float(0.75 * target_position + 0.25 * self.position)
        self.last_day = date_id
        return self.position

In [ ]:
def replay_holdout(history: pd.DataFrame, days: int) -> None:
    split = len(history) - days
    if days < 1 or split < 2:
        raise ValueError("Leave at least one training day and a one-day gap.")
    # Leave one day between fitting the forward target and scoring the holdout.
    strategy = MarketStrategy(history.iloc[:split - 1])
    holdout = history.iloc[split:]
    market = holdout["forward_returns"].to_numpy()
    cash = holdout["risk_free_rate"].to_numpy()
    if not (np.isfinite(market).all() and np.isfinite(cash).all()):
        raise ValueError("Holdout returns and risk-free rates must be finite.")

    daily_rows = pl.from_pandas(holdout).iter_slices(n_rows=1)
    positions = np.array([strategy.predict(day) for day in daily_rows])
    returns = cash + positions * (market - cash)
    print(f"{days}-day holdout | strategy {np.prod(1 + returns) - 1:.2%} "
          f"| buy and hold {np.prod(1 + market) - 1:.2%}")


history = pd.read_csv(DATA_DIR / "train.csv").sort_values("date_id")
history = history.loc[(history["date_id"] >= 37) & np.isfinite(history[TARGET])]
if HOLDOUT_DAYS:
    replay_holdout(history, HOLDOUT_DAYS)
strategy = MarketStrategy(history)

In [ ]:
sys.path.insert(0, str(DATA_DIR))
from kaggle_evaluation.default_inference_server import DefaultInferenceServer

server = DefaultInferenceServer(strategy.predict)
if os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
    server.serve()
else:
    server.run_local_gateway((str(DATA_DIR),))